# S2 α-sweep: higgs-follow

Runs the S2 sweep for **higgs-follow**: the node-level hybrid at α ∈ {0, 0.25, 0.5, 0.75, 1} plus the `mlp`
(no graph) and `gcn` baselines, 3 seeds each (`configs/sweeps/social_s2_higgs_follow.yaml`).
**Use a GPU runtime.**

- **Parallel sessions.** The four `02_s2_*` notebooks can run at the same time in separate Colab sessions
  (one per dataset): each sweep writes to its own `exp_name` under `OUT_DIR`, so no two sessions touch the
  same run folder.
- **Resume.** After a disconnect, rerun every cell from the top, then the sweep cell: finished runs are
  skipped and an interrupted run resumes from its last `checkpoint.pt` on Drive.
- The dataset is downloaded into Drive once, copied to local disk for training, and its subgraph cache is
  saved back to Drive the first time it is built.

## 1. Settings (edit)

`BRANCH` defaults to `main`; set it to another branch to run that code instead.

In [ ]:
# EDIT: settings for this notebook
BRANCH = 'social-pipeline'          # git branch to run; change it to test a feature branch
USE_TOKEN = False        # True if the repo is private (needs the GITHUB_TOKEN Colab secret, see step 3)
DRIVE = '/content/drive/MyDrive/Thesis/gnn_mech'
DATA_ROOT = f'{DRIVE}/data'    # downloads and processed caches (kept on Drive, reused by every session)
OUT_DIR = f'{DRIVE}/runs'      # run folders (kept on Drive so runs resume after a disconnect)
LOCAL_DATA = '/content/data'   # fast local copy of the dataset used during training
DATASET = 'higgs-follow'

## 2. Check the GPU (Runtime > Change runtime type > T4 GPU)

In [ ]:
!nvidia-smi

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Get the code

Clones the repo at `BRANCH`, or pulls if it is already there. If the repo is private, create a GitHub
personal access token (read access to the repo is enough), store it as a Colab secret named
`GITHUB_TOKEN` (key icon in the left sidebar) and set `USE_TOKEN = True` in step 1. Never paste the
token into the notebook itself.

In [ ]:
import os, subprocess

REPO_DIR = '/content/message-passing-in-gnns'
url = 'https://github.com/djdhawal/message-passing-in-gnns.git'
if USE_TOKEN:
    from google.colab import userdata
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/djdhawal/message-passing-in-gnns.git"

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, url, REPO_DIR], check=True)
%cd {REPO_DIR}
!git log -1 --oneline

## 5. Install

In [ ]:
!pip install -q torch_geometric
!pip install -q -e .

## 6. Folders on Drive

In [ ]:
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)
print('data on Drive:', DATA_ROOT)
print('runs on Drive:', OUT_DIR)

### Sweep file

In [ ]:
cands = ['configs/sweeps/social_s2_higgs_follow.yaml', 'configs/sweeps/social_s2_higgs-follow.yaml']
SWEEP = next((p for p in cands if os.path.exists(p)), cands[0])
print('sweep:', SWEEP, '' if os.path.exists(SWEEP) else '(MISSING: check BRANCH)')

## 7. Data helpers

`find_source_dir` finds the dataset's folder under `<root>/social/`. `sync` copies only files that are
missing or newer (rsync). Nothing to edit here.

In [ ]:
import shutil
from pathlib import Path


def _source_candidates(name):
    # Folder names under <root>/social/ that may hold `name`; gnn_mech.social.sources decides when it can.
    cands = []
    try:
        from gnn_mech.social import sources
        for attr in ('source_of', 'source_name', 'source_for'):
            fn = getattr(sources, attr, None)
            if callable(fn):
                try:
                    cands.append(str(fn(name)))
                except Exception:
                    pass
        for attr in ('SOURCES', 'SOURCE_OF', 'DATASET_SOURCE', 'DATASETS'):
            table = getattr(sources, attr, None)
            if isinstance(table, dict) and name in table:
                v = table[name]
                if not isinstance(v, str):
                    v = v.get('source') if isinstance(v, dict) else getattr(v, 'source', None)
                if v:
                    cands.append(str(v))
    except Exception:
        pass
    cands.append(name)
    if name.startswith('twitch-') and name != 'twitch-gamers':
        cands += ['musae', 'twitch', 'twitch-musae', 'musae-twitch']
    if name.startswith('higgs-'):
        cands += ['higgs', 'higgs-twitter']
    if name.startswith('reddit-'):
        cands += ['reddit', 'reddit-hyperlinks', 'soc-redditHyperlinks']
    return list(dict.fromkeys(cands))


def find_source_dir(name, root, fetched=None):
    # Name of the folder <root>/social/<source>/ that holds `name`, or None if it does not exist yet.
    social = Path(root) / 'social'
    if isinstance(fetched, (str, os.PathLike)):
        try:
            rel = Path(fetched).resolve().relative_to(social.resolve())
            if rel.parts:
                return rel.parts[0]
        except Exception:
            pass
    for c in _source_candidates(name):
        if (social / c).is_dir():
            return c
    return None


def sync(src, dst):
    # Copy files that are missing or newer in src into dst; never deletes anything.
    os.makedirs(dst, exist_ok=True)
    if shutil.which('rsync'):
        subprocess.run(['rsync', '-a', '--update', f'{src}/', f'{dst}/'], check=True)
    else:
        shutil.copytree(src, dst, dirs_exist_ok=True)

## 8. Download the dataset into Drive (once)

`gnn_mech.social.sources.fetch` downloads and parses the raw files into `DATA_ROOT/social/<source>/` and
skips files that already exist, so later sessions reuse them. If it fails, the run downloads the data
itself on first use.

In [ ]:
fetched = None
try:
    from gnn_mech.social import sources
    fetched = sources.fetch(DATASET, root=DATA_ROOT)
except Exception as e:
    print(f'[warn] fetch({DATASET!r}) failed: {e!r}')
SOURCE = find_source_dir(DATASET, DATA_ROOT, fetched)
print('source folder on Drive:', SOURCE)

## 9. Copy the dataset to local disk

Training reads from `LOCAL_DATA` (much faster than Drive). Only missing or newer files are copied.
Run outputs still go to `OUT_DIR` on Drive.

In [ ]:
if SOURCE:
    sync(f'{DATA_ROOT}/social/{SOURCE}', f'{LOCAL_DATA}/social/{SOURCE}')
    RUN_DATA = LOCAL_DATA
else:
    print('[warn] no dataset folder on Drive yet: this session reads and writes data on Drive directly')
    RUN_DATA = DATA_ROOT
COMMON = f'out_dir={OUT_DIR} data.root={RUN_DATA}'
print(COMMON)

In [ ]:
!du -sh {RUN_DATA}/social/* 2>/dev/null || echo 'nothing local yet'

## 10. Build the subgraph cache (first session only) and save it to Drive

Loads every data config used by the sweep once. The first time, this partitions the graph and writes
`social/<source>/processed/<hash>.pt`; later sessions find it and skip the work. New files are then copied
back to Drive so the next session reuses them.

In [ ]:
import gc, json
from gnn_mech.data import load_dataset

from gnn_mech.sweep import load_sweep
try:
    DATA_CFGS = [(r.name, r.cfg) for r in load_sweep(SWEEP, [f'out_dir={OUT_DIR}', f'data.root={RUN_DATA}'])[0]]
except Exception as e:
    DATA_CFGS = []
    print(f'[warn] could not read {SWEEP}: {e!r}')
seen = set()
for label, c in DATA_CFGS:
    d = c.to_dict()['data']
    d.pop('root', None)
    key = json.dumps(d, sort_keys=True, default=str)
    if key in seen:
        continue
    seen.add(key)
    try:
        tr, va, te, info = load_dataset(c.data)
        print(f'{label}: {len(tr)} / {len(va)} / {len(te)} subgraphs (train / val / test), '
              f'task {info.task}, feat_dim {getattr(info, "feat_dim", "?")}')
        del tr, va, te
    except Exception as e:
        print(f'[warn] could not prepare data for {label}: {e!r} (the run will try again)')
gc.collect()

if SOURCE is None:
    SOURCE = find_source_dir(DATASET, RUN_DATA)
if RUN_DATA != DATA_ROOT and SOURCE:
    sync(f'{RUN_DATA}/social/{SOURCE}', f'{DATA_ROOT}/social/{SOURCE}')
    print('cache saved to Drive:', f'{DATA_ROOT}/social/{SOURCE}')

## 11. Dry run: the run list, each run's status and the estimated remaining hours

In [ ]:
!python -m gnn_mech.sweep {SWEEP} --dry-run {COMMON}

## 12. Run the sweep (rerun this cell after a disconnect to resume)

In [ ]:
!python -m gnn_mech.sweep {SWEEP} {COMMON}

### Save any new data files back to Drive

In [ ]:
if RUN_DATA != DATA_ROOT:
    SOURCE = SOURCE or find_source_dir(DATASET, RUN_DATA)
    if SOURCE:
        sync(f'{RUN_DATA}/social/{SOURCE}', f'{DATA_ROOT}/social/{SOURCE}')
        print('synced to Drive')

## 13. Results so far for higgs-follow (mean ± std over seeds)

In [ ]:
import json, os
import numpy as np
import pandas as pd

PRIMARY = ('auroc', 'acc', 'r2', 'ap', 'mae', 'rmse')   # first one present is the headline metric
BASELINES = ('mlp', 'gcn')


def read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def primary_metric(split):
    keys = [k for k, v in (split or {}).items() if k != 'loss' and isinstance(v, (int, float))]
    for k in PRIMARY:
        if k in keys:
            return k
    return keys[0] if keys else None

def collect(out_dir, exps=None):
    # One row per finished run (config.json + metrics.json); unreadable or unfinished runs are skipped.
    rows = []
    root = Path(out_dir)
    if not root.is_dir():
        return pd.DataFrame()
    for cfg_path in root.glob('*/*/seed*/config.json'):
        run = cfg_path.parent
        exp = run.parent.parent.name
        if exps is not None and exp not in exps:
            continue
        cfg, m = read_json(cfg_path), read_json(run / 'metrics.json')
        if not cfg or not m or not isinstance(m.get('test'), dict):
            continue
        model, data = cfg.get('model', {}), cfg.get('data', {})
        metric = primary_metric(m['test'])
        row = {'exp': exp, 'hash': run.parent.name, 'dataset': data.get('name'), 'target': data.get('target', ''),
               'arch': model.get('arch'), 'attn': model.get('attn', 'softmax'), 'alpha': model.get('alpha'),
               'seed': cfg.get('seed'), 'metric': metric,
               'test': m['test'].get(metric), 'val': (m.get('val') or {}).get(metric),
               'best_epoch': m.get('best_epoch'), 'epochs_run': m.get('epochs_run'),
               'hours': (m.get('seconds') or 0) / 3600, 'n_params': m.get('n_params'), 'path': str(run)}
        row.update({f'test_{k}': v for k, v in m['test'].items() if isinstance(v, (int, float))})
        rows.append(row)
    return pd.DataFrame(rows)


def summarize(df):
    # dataset x arch x alpha: mean +- std over seeds and n. Baselines ignore alpha.
    if df.empty:
        return df
    df = df.copy()
    df['alpha'] = np.where(df['arch'].isin(BASELINES), np.nan, df['alpha'])
    metric_cols = [c for c in df.columns if c.startswith('test_')]
    g = df.groupby(['dataset', 'metric', 'arch', 'attn', 'alpha'], dropna=False)
    out = g.agg(n=('seed', 'nunique'), mean=('test', 'mean'), std=('test', 'std'),
                val_mean=('val', 'mean'), hours=('hours', 'sum'),
                **{f'{c}_mean': (c, 'mean') for c in metric_cols}).reset_index()
    out['std'] = out['std'].fillna(0.0)
    out['test (mean ± std)'] = [f'{a:.4f} ± {b:.4f}' for a, b in zip(out['mean'], out['std'])]
    return out.sort_values(['dataset', 'arch', 'alpha']).reset_index(drop=True)

from pathlib import Path
from gnn_mech.sweep import load_sweep

EXPS = sorted({r.cfg.exp_name for r in load_sweep(SWEEP, [f'out_dir={OUT_DIR}'])[0]})
print('experiments:', EXPS)
runs_df = collect(OUT_DIR, EXPS)
if runs_df.empty:
    print('no finished runs yet')
else:
    table = summarize(runs_df)
    display(table[['dataset', 'metric', 'arch', 'attn', 'alpha', 'n', 'test (mean ± std)', 'val_mean', 'hours']])